<a href="https://colab.research.google.com/github/ksuplee/tensorflow-nlp-tutorial/blob/main/13_AI_Agent/13_02_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 13_02 RAG 파이프라인 : 인덱싱 -> 검색 -> 생성

- 임베딩(Embedding): 문서와 질문을 고정 길이 벡터(수치 표현)로 변환함(7주차 워드 임베딩의 문장 확장판). 의미가 비슷한 텍스트는 벡터 공간에서 가깝게 위치함  
- 벡터 데이터베이스(Vector DB): 대량의 벡터를 저장하고 유사한 것을 빠르게 찾아주는 전용 저장소. FAISS(Facebook AI Similarity Search), Chroma, Milvus 등이 대표적임  
- 의미 검색(Semantic Search): 벡터 공간에서 질문과 코사인 유사도(cosine similarity)가 높은(의미가 비슷한) 문서를 찾음; ANN(Approximate Nearest Neighbor, 근사 최근접 이웃): 수백만 벡터에서도 정확한 전수 검색 대신 근사 알고리즘으로 빠르게 top-k를 찾는 기법임  
- 생성: 검색된 결과를 LLM에 입력하여 질문의 답변을 생성함  

## Hugging Face `transformers` 라이브러리를 사용하여 `llm` 정의하기

`transformers` 라이브러리를 설치해야 합니다. Colab에서는 이미 설치되어 있을 수 있습니다.

**참고**: 로컬에서 LLM을 실행하려면 많은 메모리(RAM 및 GPU)가 필요할 수 있습니다. 작은 모델을 사용하거나 Colab Pro와 같은 더 강력한 런타임을 사용해야 할 수 있습니다.

## RAG 파이프라인에 `llm` 통합하기

둘 중 하나의 `llm` 정의(예: `llm_openai` 또는 `llm_huggingface`)를 선택하여 `llm` 변수에 할당한 후, 원래 코드의 `answer = llm(prompt)` 부분을 실행하세요.

In [6]:
# transformers 설치 (아직 설치되지 않았다면 주석 해제)
# !pip install transformers accelerate

from transformers import pipeline

# 텍스트 생성을 위한 파이프라인 로드
# 더 작은 모델을 선택하거나, GPU가 충분한지 확인하세요.
# 예시로 'beomi/KoAlpaca-Polyglot-5.8B'를 사용합니다.
# 이 모델을 로드하는 데 시간이 걸릴 수 있습니다.
# model_name = "beomi/KoAlpaca-Polyglot-5.8B"
# llm_huggingface_pipe = pipeline("text-generation", model=model_name)

# 또는 더 가벼운 옵션으로 간단한 언어 모델 파이프라인을 시뮬레이션할 수 있습니다.
# 실제 LLM 호출처럼 동작하도록 하는 더미 함수를 만듭니다.
# 실제 Hugging Face 모델을 사용하려면 위 주석 처리된 부분을 사용하세요.

def call_huggingface_llm_dummy(prompt_text):
    if "트랜스포머는 언제 나왔나?" in prompt_text:
        return "트랜스포머는 2017년에 발표되었습니다."
    return f"Hugging Face 모델 시뮬레이션 응답: {prompt_text[:50]}..."

llm_huggingface = call_huggingface_llm_dummy

# 실제 Hugging Face 모델을 사용하려면, 위에서 주석 처리된 pipeline을 초기화하고
# 아래와 같이 llm_huggingface를 재정의할 수 있습니다:
# def call_huggingface_llm(prompt_text):
#     return llm_huggingface_pipe(prompt_text)[0]['generated_text']
# llm_huggingface = call_huggingface_llm

# 테스트 예시
# print(llm_huggingface("Hugging Face 모델, 안녕하세요."))

In [14]:
from sentence_transformers import SentenceTransformer, util

# ── 0) 준비: 임베딩 모델 로드 (인덱싱·검색이 공유) ──
model = SentenceTransformer("jhgan/ko-sroberta-multitask")  # 한국어 임베딩 모델

# ── ① 인덱싱 (오프라인): 문서 → 임베딩 → 벡터 저장 ──
docs = [  # 지식 베이스(문서 조각)
    "트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.",
    "BERT는 양방향 인코더로 사전학습된 모델이다.",
    "RAG는 검색한 문서를 프롬프트에 결합해 답을 생성한다.",
]
doc_emb = model.encode(docs, convert_to_tensor=True)   # 문서 임베딩 = 벡터 DB 역할

# ── ② 검색 (온라인): 질문 임베딩 → 코사인 유사도 top-k ──
query = "트랜스포머는 언제 나왔나?"
q_emb = model.encode(query, convert_to_tensor=True)    # 질문 임베딩
scores = util.cos_sim(q_emb, doc_emb)[0]               # 의미 검색(코사인 유사도)
top_idx = scores.argmax().item()                       # 가장 유사한 문서(top-1)
print("\n====== Output ======")
print(f"검색 결과: {docs[top_idx]}")
# → '트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.'

# ── ③ 생성: 검색 결과를 프롬프트에 결합 → LLM에 전달 ──
prompt = f"[문서] {docs[top_idx]}\n[질문] {query} 위 문서를 근거로 답하라."
answer = llm(prompt)   # 실제로는 LLM 호출로 근거 기반 답변 생성
print("\n====== 문서/질문/답변 ======")
print(prompt)
print("[답변] "+answer+'\n')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


====== Output ======
검색 결과: 트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.

====== 문서/질문/답변 ======
[문서] 트랜스포머는 2017년 발표된 셀프 어텐션 기반 모델이다.
[질문] 트랜스포머는 언제 나왔나? 위 문서를 근거로 답하라.
[답변] 트랜스포머는 2017년에 발표되었습니다.

